# Phase 4 — function-mode KD with fixed loss (depth12/experts80)

After PR #71 fixes `make_kd_loss_fn` (teacher/student alignment + coverage term),
re-train on the **same** function-mode teacher cache and re-eval HumanEval-TS n=40.
Prior broken-loss run scored **0.0**; this run's **best-val checkpoint (step 199)**
is the headline number — final @ 2999 regressed.

In [1]:
import json
from pathlib import Path

DEBUG = Path("../docs/debug")
PREFIX = "phase4_depth12_experts80_functions_fixedloss"
summary = json.loads((DEBUG / f"{PREFIX}_run_summary.json").read_text())
summary

{'kd_wall_clock_sec': 11443,
 'kd_wall_clock_human': '3h 10m 43s',
 'val_best_step': 199,
 'val_best': 0.3154929044066618,
 'val_final_step': 2999,
 'val_final': 0.4507074132654816,
 'humaneval_final_pass_at_1': 0.175,
 'humaneval_best_pass_at_1': 0.55}

In [2]:
import matplotlib.pyplot as plt

vals = []
for line in (DEBUG / f"{PREFIX}_kd_metrics.jsonl").read_text().splitlines():
    row = json.loads(line)
    if "val_loss" in row:
        vals.append((row["step"], row["val_loss"]))
steps, losses = zip(*vals)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(steps, losses, linewidth=1)
ax.axvline(summary["val_best_step"], color="C1", linestyle="--", label=f"best @ {summary['val_best_step']}")
ax.set_xlabel("step")
ax.set_ylabel("val KD loss")
ax.set_title("Fixed-loss function-mode KD")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<Figure size 800x400 with 1 Axes>

In [3]:
comparison = {
    "pruned_only": 0.475,
    "broken_loss_function_kd": 0.0,
    "fixed_loss_final": summary["humaneval_final_pass_at_1"],
    "fixed_loss_best": summary["humaneval_best_pass_at_1"],
}
comparison

{'pruned_only': 0.475,
 'broken_loss_function_kd': 0.0,
 'fixed_loss_final': 0.175,
 'fixed_loss_best': 0.55}

In [4]:
gens = json.loads(
    (DEBUG / f"{PREFIX}_humaneval_generations_best_multiple-ts.json").read_text()
)
for i in range(3):
    text = gens[i][0]
    print("=" * 72)
    print(f"best-val problem {i} (len={len(text)})")
    print(text[:1200])
    if len(text) > 1200:
        print("... [truncated]")

best-val problem 0 (len=556)
// ... see docs/debug/phase4_depth12_experts80_functions_fixedloss_humaneval_generations_best_problems_0-2.txt
